SMS SPAM DETECTION MODEL EVALUATION

### Import Libraries

In [105]:
# install project in editable mode so local packages (if any) are available in the notebook environment
%pip install -e .
%pip install plotly

import os  
import pandas as pd
import numpy as np 
import joblib 
import ast
import mysql.connector 

from tqdm import tqdm
from sklearn.decomposition import PCA
from sklearn.cluster import HDBSCAN
from sklearn.metrics import f1_score
from tqdm import tqdm 

import warnings
warnings.filterwarnings("ignore")

Obtaining file:///C:/Users/cj_khoh/Documents/UnifiedComms/GitHub/Project/Spam_Detection/notebooks
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: file:///C:/Users/cj_khoh/Documents/UnifiedComms/GitHub/Project/Spam_Detection/notebooks does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Load data

In [106]:
con = mysql.connector.connect(
    host = '10.168.51.196',
    port = 3306,
    user = 'unified',
    password = 'unified'
)

cur = con.cursor()

query = """
    select  	
        ml.id,
        data.payload,
        day(data.current_datetime) as date,
        hour(data.current_datetime) as hour,
        ml.embedding, 
        ml.spam_label,
        ml.spam_label_update,
        ml.confidence_score
    from sms_spam_cd.ml_spam_result ml
    inner join sms_spam_cd.data_tdr_spam_filter data on data.id = ml.id
    where hour(data.current_datetime) = 0
"""

cur.execute(query)

data = pd.DataFrame(cur.fetchall(), columns=['id', 'message', 'date', 'hour', 'embedding', 'spam_label', 'spam_label_update', 'confidence_score'])

data.head()

,id,message,date,hour,embedding,spam_label,spam_label_update,confidence_score
0,236791444,Chellum now too cool because all ready raining...,16,0,"b'[0.05234536528587341, -0.016886768862605095,...",0,0,0.93839
1,236791445,Assalamualaikum…\nKalo ada no wasap aku hang t...,16,0,"b'[0.0032701280433684587, -0.10649462044239044...",1,1,0.57340
2,236792627,Dont make fun,16,0,"b'[0.05241803452372551, -0.07558398693799973, ...",0,0,0.93631
3,236793544,12 dah,16,0,"b'[0.03576604276895523, -0.12311481684446335, ...",0,0,0.93165
4,236793760,168.5,16,0,"b'[0.029572995379567146, -0.1148369237780571, ...",0,0,0.92847


Preprocess 

In [107]:
unique_date = np.unique(data['date'].to_numpy())
unique_hour = np.unique(data['hour'].to_numpy()) 

In [108]:
# convert mysql blob data into numpy ndarray
def blob_to_numpy(blob):
    embeddings = [] 
    for row in tqdm(blob):
        embedding = row.decode('utf-8')  
        embeddings.append(ast.literal_eval(embedding)) 
        
    embeddings = np.array(embeddings)  
    return embeddings

In [109]:
vector = blob_to_numpy(data['embedding'])

  0%|          | 0/31149 [00:00<?, ?it/s]

100%|██████████| 31149/31149 [03:21<00:00, 154.25it/s]


Load Models

In [110]:
def load_models():
    model_list = {}

    for model_filepath in os.listdir('../models'):
        model_list[model_filepath] = joblib.load(os.path.join('../models', model_filepath))

    return model_list

Evaluation Process

### Overall Performance (By Central Limit Theorem)

In [111]:
def get_model_performance_samples(n_sample=100):
    model_list = load_models()
    model_results = []
    result_columns = []

    for date in unique_date:
        for hour in unique_hour:
            subdata = data[(data['date'] == date) & (data['hour'] == hour)]
             
            y_true = np.where(subdata['spam_label_update'].notna(), subdata['spam_label_update'], subdata['spam_label']) 
            
            for idx, key in enumerate(model_list):  
                result_columns.append(key + f'_{date}_{hour}')
                result = []
                for epoch in range(n_sample): 
                    random_idx = np.random.choice(len(vector), size=int(len(vector) * 0.3), replace=True)
                    random_x_test = vector[random_idx] 
                    random_y_true = y_true[random_idx]
                
                    y_pred = model_list[key].predict(random_x_test)
                    f1_metric = f1_score(random_y_true, y_pred) 
                    
                    result.append(f1_metric)
                
                model_results.append(result)
        
    model_results = np.array(model_results)
    return model_results, result_columns


In [112]:
def generate_clt_table(model_result: np.ndarray, epochs: int = 100):
    def get_random_sample_mean(data: np.ndarray, frac=0.1):
        n_sample = int(len(data) * frac)
        return np.random.choice(data, n_sample).mean()
    
    clt_table = {}
    model_results, result_columns = get_model_performance_samples(n_sample=300)
    
    for model_name, model_result in zip(result_columns, model_results):
        means = []
        for epoch in range(epochs):
            means.append(get_random_sample_mean(model_result))
        
        clt_table[model_name] = np.array(means)
    
    return pd.DataFrame(np.array(list(clt_table.values())).T, columns=list(clt_table.keys())) 

In [113]:
clt = generate_clt_table(load_models(), epochs=1000)

Plot 1: Central Limit Theorem

In [114]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

num_of_charts = len(unique_date) * len(unique_hour)
fig = make_subplots(rows=num_of_charts, cols=1)

model_list = load_models()
row = 0
for idx, model_name in enumerate(clt.columns):
    row = row + 1 if idx % len(model_list) == 0 else row
    
    fig.add_trace(
        go.Box(
            x=clt[model_name], boxpoints='all', jitter=0.5, pointpos=0, marker_opacity=0.6, name=clt.columns[idx]
        ),
        row=row, col=1
    )
 
fig.show()

### Model Performance towards Specific Category

Cluster All Data

In [115]:
pca = PCA(n_components=50)
x_reduced = pca.fit_transform(vector)

In [116]:
hdbscan = HDBSCAN()
clustering_label = hdbscan.fit_predict(x_reduced)
clustering_label

array([ -1,  -1,  -1, ...,  -1, 203,  -1], shape=(31149,))

In [117]:
labels, count = np.unique(clustering_label, return_counts=True)

Frequencies by Labels

In [118]:
df = pd.concat([pd.Series(labels.reshape(-1, 1).squeeze()), pd.Series(count.reshape(-1, 1).squeeze())], axis=1)
df.columns = ['label', 'count']

In [119]:
fig = px.bar(
    df[df.label != -1],
    x='label',
    y='count',
    text='count',
    color='count'
)

fig.show()

In [120]:
model_list = load_models()

model_accuracy = {}
model_confidence = {}

for idx, key in enumerate(model_list):
    accuracy_by_cluster = []
    confidence_by_cluster = []
    
    for label in tqdm(labels):
        subgroup_data_idx = np.where(clustering_label == label)
        subgroup_data = vector[subgroup_data_idx]
        
        y_true = data['spam_label_update'].to_numpy()[subgroup_data_idx]
        y_pred = model_list[key].predict(subgroup_data)
        
        accuracy_by_cluster.append(f1_score(y_true, y_pred, zero_division=0))
        confidence_by_cluster.append(np.mean(model_list[key].predict_proba(subgroup_data).max(axis=1)))
    
    model_accuracy[key] = accuracy_by_cluster
    model_confidence[key] = confidence_by_cluster 
        

100%|██████████| 1539/1539 [00:02<00:00, 537.11it/s]


In [121]:
target_model = 'SGDClassifier.joblib'

model_acc_diff = []
model_conf_diff = []

for key in model_list.keys():
    if key == 'SGDClassifier.joblib':
        continue
    
    model_acc_diff.append(np.array(model_accuracy[target_model]) - np.array(model_accuracy[key]))
    model_conf_diff.append(np.array(model_confidence[target_model]) - np.array(model_confidence[key]))

model_acc_diff = pd.DataFrame(model_acc_diff).T
model_conf_diff = pd.DataFrame(model_conf_diff).T

In [122]:
model_acc_diff.columns = [x for x in list(model_list.keys()) if x != 'SGDClassifier.joblib']
model_conf_diff.columns = [x for x in list(model_list.keys()) if x != 'SGDClassifier.joblib']

model_acc_diff.index = labels
model_conf_diff.index = labels


In [123]:
model_acc_diff

,SGDClassifier-202510291603.joblib
-1,0.002115
0,0.000000
1,0.000000
2,0.000000
3,0.000000
...,...
1533,0.000000
1534,0.444444
1535,0.000000
1536,0.000000


In [124]:
fig = make_subplots(rows=len(model_list) - 1, cols=1)

for i in range(len(model_list) - 1):
    fig.add_trace(
        go.Bar(x=model_acc_diff.index, y=model_acc_diff.iloc[:, i], marker_color=['green' if x >= 0 else 'red' for x in model_acc_diff.iloc[:, i]]),
        row=i+1,
        col=1
    )

fig.show()

- Red - latest model prediction changed from 'Spam' to 'Not Spam'
- Green - latest model prediction changed from 'Not Spam' to 'Spam'

In [125]:
fig = make_subplots(rows=len(model_list) - 1, cols=1)

for i in range(len(model_list) - 1):
    fig.add_trace(
        go.Bar(x=model_conf_diff.index, y=model_conf_diff.iloc[:, i], marker_color=['green' if x >= 0 else 'red' for x in model_conf_diff.iloc[:, i]]),
        row=i+1,
        col=1
    )

fig.show()

- Red - Confidence score decrease
- Green - Confidence score increase

In [138]:
idx = np.where(clustering_label == 463)
data['message'].iloc[idx]

1268           1763, Jalan Jambu 24\n81400 Senai 柔佛州\n马来西亚
5966     9-3-9, Lintang P.Ramlee, Jalan Rawang, Melodi ...
6414     1785, Lor Sibuga 13, 90000 Sandakan, Sabah, Ma...
6433     1785, Lor Sibuga 13, 90000 Sandakan, Sabah, Ma...
10126        Jalan Entiba\n93050 Sarikei Sarawak\nMalaysia
23419    181, Lorong Ramin 5, 90000 Sandakan, Sabah, Ma...
23420    181, Lorong Ramin 5, 90000 Sandakan, Sabah, Ma...
Name: message, dtype: object